In [2]:
import warnings
import pandas as pd
import gtfs_kit as gk

In [3]:
from pathlib import Path
import re

gtfs_output_path = Path("/home/simpal/otp/data")

gtfs_year = 2025
gtfs_root = Path("/home/simpal/O/sharing-trans-data/GTFS Data/CLEAN - GTFS DATA/" + str(gtfs_year))

if not gtfs_root.is_dir():
    print("INPUT ERROR: Directory not found: " + str(gtfs_root))
# Recursively find zip files
gtfs_files = list(gtfs_root.rglob("*.zip"))

# Extract YYYYMMDD from filename
def extract_date(path):
    match = re.search(r"\d{8}", path.name)
    return pd.to_datetime(match.group(), format="%Y%m%d") if match else None


gtfs_release = (
    pd.DataFrame({
        "path": gtfs_files,
        "file": [p.name for p in gtfs_files],
        "date": [extract_date(p) for p in gtfs_files],
    })
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
    .assign(
        date_end = lambda df: df["date"].shift(-1),
        date_days = lambda df: (df["date"] - pd.Timestamp("1970-01-01")).dt.days,
        date_end_days = lambda df: (df["date_end"] - pd.Timestamp("1970-01-01")).dt.days,
    )
)


print(f"Total number GTFS files: {len(gtfs_release)}")

Total number GTFS files: 24


In [4]:
gtfs_list = {}
#Read all gtfs_files
for _, row in gtfs_release.iterrows():
    print(f"Reading GTFS file: {row['file']}")

    feed = gk.feed.read_feed(row["path"], dist_units = "m") #Not sure what dist_unit is.

    gtfs_list[row["file"]] = feed


Reading GTFS file: GTFS_20250102.zip
Reading GTFS file: GTFS_20250113.zip
Reading GTFS file: GTFS_20250127.zip
Reading GTFS file: GTFS_20250210.zip
Reading GTFS file: GTFS_20250224.zip
Reading GTFS file: GTFS_20250310.zip
Reading GTFS file: GTFS_20250324.zip
Reading GTFS file: GTFS_20250407.zip
Reading GTFS file: GTFS_20250422.zip
Reading GTFS file: GTFS_20250505.zip
Reading GTFS file: GTFS_20250519.zip
Reading GTFS file: GTFS_20250616.zip
Reading GTFS file: GTFS_20250630.zip
Reading GTFS file: GTFS_20250728.zip
Reading GTFS file: GTFS_20250811.zip
Reading GTFS file: GTFS_20250825.zip
Reading GTFS file: GTFS_20250908.zip
Reading GTFS file: GTFS_20250922.zip
Reading GTFS file: GTFS_20251006.zip
Reading GTFS file: GTFS_20251020.zip
Reading GTFS file: GTFS_20251103.zip
Reading GTFS file: GTFS_20251117.zip
Reading GTFS file: GTFS_20251201.zip
Reading GTFS file: GTFS_20251215.zip


In [5]:
#Function to trunceate GTFS feed at specified date
#Cutoff date will not be included
def truncate_feed_to_date(feed_i, cutoff_date):
    cutoff_str = cutoff_date.strftime("%Y%m%d")

    if feed_i.calendar is not None:
        cal = feed_i.calendar.copy()
        cal = cal[cal.start_date < cutoff_str]
        cal.loc[cal.end_date >= cutoff_str, "end_date"] = cutoff_str
        feed_i.calendar = cal
        del cal

    if feed_i.calendar_dates is not None:
        cd = feed_i.calendar_dates.copy()
        cd = cd[cd.date < cutoff_str]
        feed_i.calendar_dates = cd
        del cd

    #filter trips using valid service_id
    valid_service_ids = set()
    if feed_i.calendar is not None:
        valid_service_ids.update(feed_i.calendar.service_id.unique())

    if feed_i.calendar_dates is not None:
        valid_service_ids.update(feed_i.calendar_dates.service_id.unique())

    feed_i.trips = feed_i.trips[
        feed_i.trips.service_id.isin(valid_service_ids)
    ]

    #restrict_to_trips: Build a new feed by restricting this one to only the stops, trips, shapes, etc. used by the trips of the given IDs. Return the resulting feed.
    feed_trunc = gk.miscellany.restrict_to_trips(feed_i, feed_i.trips.trip_id.tolist())


    return feed_trunc

In [ ]:
for i, row in gtfs_release.iterrows():
    file_key = row["file"]
    cutoff = row["date_end"]

    if pd.isna(cutoff):
        continue

    print(f"Truncating {file_key} to {cutoff.date()}")
    gtfs_list[file_key] = truncate_feed_to_date(gtfs_list[file_key], cutoff)


Truncating GTFS_20250102.zip to 2025-01-13
Truncating GTFS_20250113.zip to 2025-01-27
Truncating GTFS_20250127.zip to 2025-02-10
Truncating GTFS_20250210.zip to 2025-02-24


In [ ]:
# Configuration: ID columns and where they appear as foreign keys
ID_CONFIG = {
    "agency_id": {
      "primary_table": "agency",
      "identity_cols": ["agency_name", "agency_timezone"],
      "foreign_keys": [
          ("routes", "agency_id"),
      ]
    },
    "route_id": {
        "primary_table": "routes",
        "identity_cols": ["agency_id", "route_short_name", "route_type"],
        "foreign_keys": [
            ("trips", "route_id"),
            ("transfers", "from_route_id"),
            ("transfers", "to_route_id")
        ]
    },
    "stop_id": {
        "primary_table": "stops",
        "identity_cols": ["stop_lat", "stop_lon"],
        "foreign_keys": [
            ("stop_times", "stop_id"),
            ("transfers", "from_stop_id"),
            ("transfers", "to_stop_id"),
        ]
    },
    "shape_id": {
        "primary_table": "shapes",
        "identity_cols": ["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"],
        "foreign_keys": [
            ("trips", "shape_id"),
        ]
    },
    "trip_id": {
        "primary_table": "stop_times",
        "identity_cols": ["stop_id", "arrival_time", "departure_time", "stop_sequence"],
        "foreign_keys": [
            ("trips", "trip_id"),
            ("transfers", "from_trip_id"),
            ("transfers", "to_trip_id"),
        ]
    },
#    "trip_id": {
#        "primary_table": "trips",
#        "identity_cols": ["service_id", "trip_headsign", "trip_short_name", "direction_id"],
#        "foreign_keys": [
#            ("stop_times", "trip_id"),
#        ]
#    },
}

#
ID_CONFIG_service_id = {
    "service_id": {
      "primary_table": "calendar",
      "identity_cols": ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday", "start_date", "end_date"],
      "foreign_keys": [
          ("trips", "service_id"),
          ("calendar_dates", "service_id"),
      ]
    }
}
ID_CONFIG_prefix = {**ID_CONFIG, **ID_CONFIG_service_id}

In [ ]:
print("\nMerging all feeds into combined GTFS feed...")

feed_names = list(gtfs_list.keys())
combined_feed = gtfs_list[feed_names[0]]
print(f"Starting with base feed: {feed_names[0]}")

for feed_name in feed_names[1:]:
    print(f"Merging feed: {feed_name}")
    feed_to_merge = gtfs_list[feed_name]

    combined_feed.agency = pd.concat([combined_feed.agency, feed_to_merge.agency.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.routes = pd.concat([combined_feed.routes, feed_to_merge.routes.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.stops = pd.concat([combined_feed.stops, feed_to_merge.stops.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.trips = pd.concat([combined_feed.trips, feed_to_merge.trips.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.stop_times = pd.concat([combined_feed.stop_times, feed_to_merge.stop_times.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.calendar = pd.concat([combined_feed.calendar, feed_to_merge.calendar.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.calendar_dates = pd.concat([combined_feed.calendar_dates, feed_to_merge.calendar_dates.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.shapes = pd.concat([combined_feed.shapes, feed_to_merge.shapes.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.transfers = pd.concat([combined_feed.transfers, feed_to_merge.transfers.assign(feed_id = feed_name)], ignore_index=True)

#    del gtfs_list[feed_name]
    del feed_to_merge


print(f"\nMerge complete! Combined feed statistics:")
if combined_feed.agency is not None:
    print(f"  Agencies: {len(combined_feed.agency)}")
if combined_feed.routes is not None:
    print(f"  Routes: {len(combined_feed.routes)}")
if combined_feed.stops is not None:
    print(f"  Stops: {len(combined_feed.stops)}")
if combined_feed.trips is not None:
    print(f"  Trips: {len(combined_feed.trips)}")
if combined_feed.stop_times is not None:
    print(f"  Stop times: {len(combined_feed.stop_times)}")
if combined_feed.calendar is not None:
    print(f"  Calendar entries: {len(combined_feed.calendar)}")
if combined_feed.shapes is not None:
    print(f"  Shape points: {len(combined_feed.shapes)}")


In [ ]:
def deduplicate_feed(feed, id_col, primary_table, identity_cols, foreign_keys):

    df = getattr(feed, primary_table, None)
    if df is None or df.empty:
        raise ValueError(f"Error: {primary_table} not found in feed")

    initial_count = len(df)

    # Filter identity_cols to those present in the dataframe
    use_identity_cols = [c for c in identity_cols if c in df.columns and c != id_col]

    if not use_identity_cols:
        print(f"  Warning: No identity columns found for {primary_table}")
        return 0

    # Check if this is a sequence-based table (shapes, stop_times have multiple rows per ID)
    sequence_cols = ['shape_pt_sequence', 'stop_sequence']
    is_sequence_table = any(col in df.columns for col in sequence_cols)

    if is_sequence_table:
        # For sequence tables: create signature from all rows grouped by ID
        sort_col = next((c for c in sequence_cols if c in df.columns), None) #sequence column
        df_sorted = df.sort_values([id_col, sort_col])
        
        # Create row-level signature by concatenating identity columns
        df_sorted['_row_sig'] = df_sorted[use_identity_cols].astype(str).agg('|'.join, axis=1)
        
        # Group and concatenate row signatures into single signature per ID
        signatures = (
            df_sorted
            .groupby(id_col)['_row_sig']
            .agg(lambda x: '||'.join(x))
            .reset_index(name='_signature')
        )

        # Map signature to canonical (minimum) ID
        canonical_map = (
            signatures
            .groupby('_signature')[id_col]
            .min()
            .to_dict()
        )

        # Create ID to canonical ID mapping
        id_to_canonical = (
            signatures
            .set_index(id_col)['_signature']
            .map(canonical_map)
            .to_dict()
        )

        # Get set of canonical IDs
        canonical_ids = set(canonical_map.values())

        # Update primary table: keep only rows with canonical IDs
        setattr(feed, primary_table, df[df[id_col].isin(canonical_ids)].reset_index(drop=True))

    else:
        # For simple tables: group by identity columns directly
        df_sorted = df.sort_values(id_col)  # Ensure deterministic selection

        # Map each unique combination of identity cols to canonical (minimum) ID
        canonical_df = (
            df_sorted
            .groupby(use_identity_cols, dropna=False)[id_col]
            .first()  # First after sort = minimum
            .reset_index()
        )

        # Create mapping from all IDs to canonical IDs
        id_to_canonical = (
            df_sorted
            .merge(canonical_df, on=use_identity_cols, suffixes=('', '_canonical'))
            .set_index(id_col)[f'{id_col}_canonical']
            .to_dict()
        )

        # Update primary table: keep only canonical rows
        canonical_ids = set(canonical_df[id_col])
        setattr(feed, primary_table, df[df[id_col].isin(canonical_ids)].reset_index(drop=True))

    # Update all foreign key references
    for fk_table, fk_col in foreign_keys:
        fk_df = getattr(feed, fk_table, None)
        if fk_df is None or fk_col not in fk_df.columns:
            raise KeyError(f"Error: Foreign key column {fk_col} not found in {fk_table}")

        # Map foreign keys to canonical IDs
        fk_df[fk_col] = fk_df[fk_col].map(lambda x: id_to_canonical.get(x, x) if pd.notna(x) else x)
        setattr(feed, fk_table, fk_df)

    final_count = len(getattr(feed, primary_table))
    duplicates_removed = initial_count - final_count

    return duplicates_removed


print("Starting deduplication process...")
print(f"\nBefore deduplication:")
for table in ['stops', 'stop_times', 'shapes', 'routes', 'agency']:
    df = getattr(combined_feed, table, None)
    if df is not None:
        print(f"  {table}: {len(df)}")

# Apply deduplication for each ID type
for id_col, config in ID_CONFIG.items():
    print(f"\nDeduplicating {id_col}...")

    removed = deduplicate_feed(
        combined_feed,
        id_col,
        config["primary_table"],
        config["identity_cols"],
        config["foreign_keys"]
    )

    df = getattr(combined_feed, config["primary_table"], None)
    if df is not None:
        print(f"  {config['primary_table']}: removed {removed} duplicates, {len(df)} remaining")

print("\n" + "=" * 50)
print("Deduplication complete!")
print(f"\nAfter deduplication:")
for table in ['stops', 'stop_times', 'shapes', 'routes', 'agency']:
    df = getattr(combined_feed, table, None)
    if df is not None:
        print(f"  {table}: {len(df)}")

Starting deduplication process...

Before deduplication:
  stops: 892866

Deduplicating service_id...
  calendar: removed 37723 duplicates, 24 remaining

Deduplicating agency_id...
  agency: removed 0 duplicates, 479 remaining

Deduplicating route_id...
  routes: removed 3226 duplicates, 35417 remaining

Deduplicating stop_id...
  stops: removed 23830 duplicates, 869036 remaining

Deduplicating shape_id...


In [ ]:
def find_conflicting_ids(gtfs_list, id_col, primary_table, identity_cols):
    """
    Identifies conflicting IDs across(!) multiple GTFS feeds based on the provided identity columns.

    This function examines the primary table within each GTFS feed to detect IDs that have
    multiple unique definitions by evaluating specified identity columns. Conflicting IDs
    are determined if an ID has more than one unique combination of identity column values
    across the feeds.

    :param gtfs_list: Dictionary of GTFS feeds, where keys are feed names and values are feed objects.
    :param id_col: Column name representing the unique identifier in the primary table.
    :param primary_table: Name of the table in the GTFS feed to be analyzed for conflicts.
    :param identity_cols: List of column names to check for unique combinations.
    :return: A set containing IDs with conflicting definitions.
    """
    all_records = []
    for feed_name, feed in gtfs_list.items():
        df = getattr(feed, primary_table, None)

        if df is None:
            warnings.warn(f"Warning: {primary_table} not found in feed: {feed_name}")
            continue
        if not id_col in df.columns:
            raise KeyError(f"Error: id_col: {id_col}, not column in {primary_table} of feed: {feed_name}")


        use_identity_cols = [c for c in identity_cols if c in df.columns]
        if len(use_identity_cols) < len(identity_cols):
            warnings.warn(f"Warning: Missing columns in {primary_table}. Only {use_identity_cols} found, but expected (identity_cols): {identity_cols}")
        use_identity_cols = use_identity_cols + [id_col]

        records = df[use_identity_cols].drop_duplicates().assign(feed_name=feed_name)
        all_records.append(records)

    if not all_records:
        return set()

    combined = pd.concat(all_records, ignore_index=True)
    use_identity_cols = [c for c in identity_cols if c in combined.columns]
    use_identity_cols = use_identity_cols + [id_col]

    # Group by ID and count unique definitions
    id_definitions = (
        combined
        .drop_duplicates(subset=use_identity_cols)
        .groupby(id_col)
        .size()
    )
    conflicting = set(id_definitions[id_definitions > 1].index)
    return conflicting


def apply_prefix_to_feeds(gtfs_list, id_col, conflicting_ids, foreign_keys, primary_table):
    """
    Applies prefixes to IDs in multiple GTFS feeds to resolve conflicts.

    This function updates the primary and foreign key columns across
    specified GTFS tables for each feed in the provided list. It only
    processes IDs listed in `conflicting_ids`, adding a prefix based
    on the respective feed name. The function is particularly useful
    in scenarios where feeds share conflicting ID values that need
    to remain unique across the dataset.

    :param gtfs_list:
        A dictionary where keys are feed names (strings), and values
        are GTFS feed objects. Each GTFS feed object should expose
        tables as attributes containing dataframes.
    :param id_col:
        A string representing the primary key column name for the
        table to be updated. For example, "trip_id" or "service_id".
    :param conflicting_ids:
        A set of ID values (strings or integers) that are found to be
        conflicting across multiple feeds and need to be resolved
        by applying a prefix.
    :param foreign_keys:
        A list of tuples, where each tuple contains the name of a
        table (string) and the column (string) representing a foreign
        key referencing the primary key.
    :return:
        None. The function modifies the GTFS feed objects in-place,
        applying prefixes to the specified columns where conflicts
        are detected.
    """
    if not conflicting_ids:
        return

    for feed_name, feed in gtfs_list.items():

        # All tables/columns to update (primary + foreign keys)
        tables_to_update = [(primary_table, id_col)]
        tables_to_update.extend(foreign_keys)

        # Collect all conflicting IDs that exist in this feed
        feed_conflicting_ids = set()

        for table_name, col_name in tables_to_update:
            df = getattr(feed, table_name, None)
            if df is None or col_name not in df.columns:
                continue

            existing_ids = set(df[col_name].dropna().unique())
            feed_conflicting_ids.update(existing_ids & conflicting_ids)

        if not feed_conflicting_ids:
            continue

        prefix = feed_name.replace("GTFS_","").replace(".zip", "") + "_"

        # Apply prefix to all tables
        for table_name, col_name in tables_to_update:
            df = getattr(feed, table_name, None)
            if df is None or col_name not in df.columns:
                continue

            mask = df[col_name].isin(feed_conflicting_ids)
            if mask.any():
                df.loc[mask, col_name] = prefix + df.loc[mask, col_name].astype(str)


# Apply to all ID types
for id_col, config in ID_CONFIG_prefix.items():
    print(f"\nProcessing {id_col}...")
    conflicting = find_conflicting_ids(
        gtfs_list,
        id_col,
        config["primary_table"],
        config["identity_cols"]
    )
    print(f"  Found {len(conflicting)} conflicting {id_col} values")

    if conflicting:
        apply_prefix_to_feeds(gtfs_list, id_col, conflicting, config["foreign_keys"], config["primary_table"])
        print(f"    Prefixed conflicting {id_col} in all feeds")